# LLM Reads Web Page and Answers Questions

In [22]:
%%time
%pip install -U requests beautifulsoup4 transformers torch ipython-autotime ipywidgets

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
CPU times: user 8.17 ms, sys: 11.7 ms, total: 19.9 ms
Wall time: 1.14 s
time: 1.14 s (started: 2025-04-16 19:35:25 -07:00)


In [23]:
%%time
import warnings
warnings.filterwarnings("ignore")

CPU times: user 15 μs, sys: 9 μs, total: 24 μs
Wall time: 25.3 μs
time: 360 μs (started: 2025-04-16 19:35:26 -07:00)


## Scrape web page content

In [24]:
%%time
import requests
from bs4 import BeautifulSoup

def fetch_webpage_content(url):
    try:
        # Send HTTP request
        headers = {'User-Agent': 'Mozilla/5.0'}  # Avoid bot detection
        response = requests.get(url, headers=headers, verify=False)
        response.raise_for_status()  # Check for request errors

        # Parse HTML with BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract text from paragraphs (customize as needed)
        paragraphs = soup.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li'])
        content = ' '.join(element.get_text(strip=True) for element in paragraphs)
        
        # Clean up: remove excessive whitespace
        content = ' '.join(content.split())

        max_length = 2000 # Limit to ~2000 chars to fit within LLM token limits
        if len(content) > max_length:
            content = content[:max_length] + '...'
        
        return content
    except requests.exceptions.SSLError as ssl_err:
        print(f"SSL Error: {ssl_err}. Try updating certifi or checking network settings.")
        return None
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

CPU times: user 7 μs, sys: 14 μs, total: 21 μs
Wall time: 23.1 μs
time: 1.19 ms (started: 2025-04-16 19:35:26 -07:00)


#### Test function 'scrape_web_page'

In [25]:
%%time
fetch_webpage_content("https://en.wikipedia.org/wiki/Artificial_intelligence")

CPU times: user 275 ms, sys: 13.9 ms, total: 289 ms
Wall time: 778 ms


'Main page Contents Current events Random article About Wikipedia Contact us Help Learn to edit Community portal Recent changes Upload file Special pages Donate Create account Log in Donate Create account Log in Contributions Talk Contents (Top) 1GoalsToggle Goals subsection1.1Reasoning and problem-solving1.2Knowledge representation1.3Planning and decision-making1.4Learning1.5Natural language processing1.6Perception1.7Social intelligence1.8General intelligence 1.1Reasoning and problem-solving 1.2Knowledge representation 1.3Planning and decision-making 1.4Learning 1.5Natural language processing 1.6Perception 1.7Social intelligence 1.8General intelligence 2TechniquesToggle Techniques subsection2.1Search and optimization2.1.1State space search2.1.2Local search2.2Logic2.3Probabilistic methods for uncertain reasoning2.4Classifiers and statistical learning methods2.5Artificial neural networks2.6Deep learning2.7GPT2.8Hardware and software 2.1Search and optimization2.1.1State space search2.1.2

time: 779 ms (started: 2025-04-16 19:35:26 -07:00)


## Initialize LLM for question answering

- Run ollama service using the script# https://github.com/krishnamanchikalapudi/developer.info/blob/develop/LLM/ollama.sh

In [26]:

# ollama run llama3.2:1b. refer execution script https://github.com/krishnamanchikalapudi/developer.info/blob/develop/LLM/ollama.sh
OLLAMA_API_URL = "http://localhost:11434/api/generate"

model_name = "llama3.2:1b"
# "Accept: application/json" -X POST ${OLLAMA_URL}/generate -d "{\"model\": \"${MODEL_NAME}\", \"prompt\":\"Why is the sky blue?\", \"raw\": true, \"stream\": false }"  

def answer_question(context, question):
    try: 
        prompt_str = f"{context}\n\nQ: {question}\nA:"
        payload = {
            "model": model_name,
            "prompt": prompt_str ,
            "raw": True,
            "stream": False
        }
        response = requests.post(OLLAMA_API_URL, json=payload)
        # {"model":"llama3.2:1b","created_at":"2025-04-17T02:29:52.925511Z","message":{"role":"assistant","content":"I was created by Meta AI, a team of researchers and engineers at Meta Platforms, Inc."},"done_reason":"stop","done":true,"total_duration":270874958,"load_duration":11150833,"prompt_eval_count":30,"prompt_eval_duration":27000000,"eval_count":20,"eval_duration":231000000}


        result = response.json()
        llm_response = result.get("response", "No response from model.")   

        print(f"LLM Response: {llm_response}")
        return llm_response
    except Exception as e:
        return f"Error processing question: {str(e)}"

time: 337 μs (started: 2025-04-16 19:35:27 -07:00)


## End-To-End testing

In [ ]:
%%time

sample_url="https://finance.yahoo.com/news/nvidia-stock-dives-as-chipmaker-sees-55-billion-hit-from-surprise-china-chip-controls-130319576.html"
question="What is the reason for Nvidia's stock dive?"

def main():
    # Fetch content from the webpage
    webpage_content = fetch_webpage_content(sample_url)
    if webpage_content.startswith("Error"):
        print(webpage_content)
        return
    
    print(f"\nQuestion: {question}")
    answer = answer_question(webpage_content, question)
    print(f"Answer: {answer}")

if __name__ == "__main__":
    main()


Question: What is the reason for Nvidia's stock dive?
LLM Response:  The stocks has dropped on its first day after CEO Jensen Huang took over.
Answer:  The stocks has dropped on its first day after CEO Jensen Huang took over.
CPU times: user 81.6 ms, sys: 8.76 ms, total: 90.3 ms
Wall time: 1.69 s
time: 1.69 s (started: 2025-04-16 19:35:27 -07:00)
